# Functional Programming in Big Data and Real World Application

## Tujuan Pembelajaran
Mahasiswa mampu menerapkan konsep pemrograman fungsional dalam konteks big data dan aplikasi dunia nyata, serta memahami keunggulan pendekatan fungsional dalam skenario produksi.

---

## Outline Materi
1. Apache Spark dan Functional Programming
2. MapReduce Paradigm
3. Stream Processing dengan Functional Approach
4. Data Pipeline Design
5. Real World Case Studies
6. Best Practices dalam Production

---
## 1. Apache Spark dan Functional Programming

### Mengapa Functional Programming untuk Big Data?

**Keuntungan FP dalam Big Data:**
- **Immutability**: Data tidak berubah, cocok untuk distributed computing
- **Pure Functions**: Mudah di-parallelize tanpa side effects
- **Higher-Order Functions**: Abstraksi operasi data yang powerful
- **Lazy Evaluation**: Optimisasi query execution plan

### Apache Spark Architecture

Apache Spark adalah framework distributed computing yang sangat bergantung pada konsep FP:
- **RDD (Resilient Distributed Dataset)**: Immutable distributed collection
- **Transformations**: Lazy operations (map, filter, flatMap)
- **Actions**: Eager operations (collect, count, reduce)

In [1]:
# Setup: Install PySpar
!pip install pyspark


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.4/455.4 MB 463.9 kB/s eta 0:00:0000:0100:02
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.0/203.0 kB 2.8 MB/s eta 0:00:00a 0:00:01
  Created wheel for pyspark: filename=pyspark-4.1.1-py2.py3-none-any.whl size=456008704 sha256=a06d8b51073e0a8b2aea5d41ef2d8a8b3a05645eb46e606ac92b3fd3e153ecf0
  Stored in directory: /Users/mac/Library/Caches/pip/wheels/16/33/a9/f8bff354a182417214933df74dace2a34b02c3e5643e8fac74
Successfully built pyspark

[notice] A new release of pip is available: 23.1.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count, sum as spark_sum, expr
from functools import reduce
from operator import add

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("FunctionalProgrammingBigData") \
    .master("local[*]") \
    .getOrCreate()

print(f"Spark Version: {spark.version}")
print(f"Spark Master: {spark.sparkContext.master}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/18 09:58:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Version: 4.1.1
Spark Master: local[*]


### Contoh: RDD Operations dengan Functional Approach

In [3]:
# Membuat RDD dari list
data = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
rdd = spark.sparkContext.parallelize(data)

# TRANSFORMATIONS (Lazy - tidak dieksekusi sampai action dipanggil)

# 1. map: Transform setiap element
squared_rdd = rdd.map(lambda x: x ** 2)

# 2. filter: Filter berdasarkan kondisi
even_rdd = rdd.filter(lambda x: x % 2 == 0)

# 3. flatMap: Map kemudian flatten
words_rdd = spark.sparkContext.parallelize(["hello world", "big data"])
flat_words = words_rdd.flatMap(lambda line: line.split())

# ACTIONS (Eager - trigger execution)
print("Original data:", rdd.collect())
print("Squared:", squared_rdd.collect())
print("Even numbers:", even_rdd.collect())
print("Flat words:", flat_words.collect())
print("Sum using reduce:", rdd.reduce(lambda a, b: a + b))

Original data: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


Squared: [1, 4, 9, 16, 25, 36, 49, 64, 81, 100]


Even numbers: [2, 4, 6, 8, 10]


Flat words: ['hello', 'world', 'big', 'data']
Sum using reduce: 55


### Lazy Evaluation dan Query Optimization

In [4]:
# Demonstration of Lazy Evaluation
print("=== Lazy Evaluation Demo ===")

# Transformations tidak dieksekusi langsung
lazy_rdd = rdd.map(lambda x: x * 2).filter(lambda x: x > 10)
print("Transformations defined, but not executed yet")

# Action memicu eksekusi seluruh DAG (Directed Acyclic Graph)
result = lazy_rdd.collect()
print(f"After action (collect): {result}")

# Spark membangun execution plan yang optimal
print("\n=== Execution Plan ===")
print(lazy_rdd.toDebugString().decode('utf-8'))

=== Lazy Evaluation Demo ===
Transformations defined, but not executed yet


After action (collect): [12, 14, 16, 18, 20]

=== Execution Plan ===
(4) PythonRDD[6] at collect at /var/folders/k3/03k08prs08l1tb44r35bvtrm0000gn/T/ipykernel_2257/3457333280.py:9 []
 |  ParallelCollectionRDD[0] at readRDDFromFile at PythonRDD.scala:299 []


---
## 2. MapReduce Paradigm

### Konsep MapReduce

MapReduce adalah programming model untuk processing large datasets dengan parallel, distributed algorithm:

1. **Map**: Transform input menjadi key-value pairs
2. **Shuffle**: Group semua values dengan key yang sama
3. **Reduce**: Aggregate values untuk setiap key

### MapReduce sebagai Functional Pattern
- Map → Higher-order function
- Reduce → Fold/aggregate operation
- Immutable data flow

In [5]:
# Classic MapReduce Example: Word Count

text_data = [
    "functional programming is powerful",
    "big data processing with functional approach",
    "functional programming enables parallel processing"
]

# Membuat RDD
text_rdd = spark.sparkContext.parallelize(text_data)

# MAP phase: transform text to (word, 1) pairs
word_pairs = text_rdd.flatMap(lambda line: line.split()) \
                      .map(lambda word: (word.lower(), 1))

# REDUCE phase: sum counts for each word
word_counts = word_pairs.reduceByKey(lambda a, b: a + b)

# Collect results
print("=== Word Count Results ===")
for word, count in sorted(word_counts.collect(), key=lambda x: x[1], reverse=True):
    print(f"{word}: {count}")

=== Word Count Results ===


functional: 3
programming: 2
processing: 2
powerful: 1
big: 1
parallel: 1
is: 1
with: 1
approach: 1
enables: 1
data: 1


### Advanced MapReduce: Log Analysis

In [6]:
# Simulating log data
log_data = [
    "2024-01-15 10:23:45 ERROR Database connection failed",
    "2024-01-15 10:24:12 INFO User login successful",
    "2024-01-15 10:25:33 ERROR Timeout exception",
    "2024-01-15 10:26:01 WARNING Low memory",
    "2024-01-15 10:27:22 ERROR File not found",
    "2024-01-15 10:28:45 INFO Request processed",
]

logs_rdd = spark.sparkContext.parallelize(log_data)

# MAP: Extract log level
log_levels = logs_rdd.map(lambda log: log.split()[2])

# REDUCE: Count by log level
level_counts = log_levels.map(lambda level: (level, 1)) \
                         .reduceByKey(add)

print("\n=== Log Level Analysis ===")
for level, count in level_counts.collect():
    print(f"{level}: {count}")

# Advanced: Filter and analyze ERROR logs
error_logs = logs_rdd.filter(lambda log: "ERROR" in log) \
                     .map(lambda log: log.split(maxsplit=3)[3])

print("\n=== Error Messages ===")
for error in error_logs.collect():
    print(f"- {error}")


=== Log Level Analysis ===


ERROR: 3
INFO: 2

=== Error Messages ===
- Database connection failed
- Timeout exception
- File not found


---
## 3. Stream Processing dengan Functional Approach

### Streaming Data Processing

Stream processing adalah processing data secara continuous, real-time. Functional programming sangat cocok karena:
- Stateless transformations
- Composable operations
- Immutable event streams

### Spark Structured Streaming

In [7]:
# Simulating streaming data dengan batch processing
import time
from datetime import datetime

# Membuat sample streaming data (sensor readings)
def generate_sensor_data(batch_id):
    """Generate sample IoT sensor data"""
    import random
    data = []
    for _ in range(5):
        data.append({
            'timestamp': datetime.now().isoformat(),
            'sensor_id': f'SENSOR_{random.randint(1, 5)}',
            'temperature': round(random.uniform(20.0, 30.0), 2),
            'humidity': round(random.uniform(40.0, 80.0), 2)
        })
    return data

# Simulate 3 batches of streaming data
print("=== Simulated Stream Processing ===\n")

all_readings = []
for batch_id in range(3):
    batch_data = generate_sensor_data(batch_id)
    all_readings.extend(batch_data)
    
    # Create DataFrame from batch
    df = spark.createDataFrame(batch_data)
    
    # Functional transformations
    processed = df.select(
        col('sensor_id'),
        col('temperature'),
        col('humidity'),
        expr('temperature * 9/5 + 32').alias('temp_fahrenheit')
    ).filter(col('temperature') > 25)
    
    print(f"Batch {batch_id}:")
    processed.show(truncate=False)
    
    time.sleep(0.5)

=== Simulated Stream Processing ===

Batch 0:
+---------+-----------+--------+---------------+
|sensor_id|temperature|humidity|temp_fahrenheit|
+---------+-----------+--------+---------------+
|SENSOR_4 |25.42      |55.09   |77.756         |
|SENSOR_5 |27.98      |52.95   |82.364         |
+---------+-----------+--------+---------------+

Batch 1:
+---------+-----------+--------+---------------+
|sensor_id|temperature|humidity|temp_fahrenheit|
+---------+-----------+--------+---------------+
|SENSOR_4 |25.79      |49.61   |78.422         |
|SENSOR_1 |28.77      |60.29   |83.786         |
|SENSOR_4 |25.33      |65.86   |77.594         |
+---------+-----------+--------+---------------+

Batch 2:
+---------+-----------+--------+---------------+
|sensor_id|temperature|humidity|temp_fahrenheit|
+---------+-----------+--------+---------------+
|SENSOR_3 |26.0       |43.67   |78.8           |
|SENSOR_4 |27.41      |70.05   |81.338         |
+---------+-----------+--------+---------------+



### Windowed Aggregations (Functional Pattern)

In [8]:
# Aggregate streaming data per sensor
from pyspark.sql.functions import round as spark_round

all_readings_df = spark.createDataFrame(all_readings)

# Functional aggregation pipeline
sensor_stats = all_readings_df.groupBy('sensor_id').agg(
    spark_round(avg('temperature'), 2).alias('avg_temp'),
    spark_round(avg('humidity'), 2).alias('avg_humidity'),
    count('*').alias('reading_count')
).orderBy('sensor_id')

print("\n=== Sensor Statistics (Aggregated) ===")
sensor_stats.show()

# Functional composition: chaining operations
high_temp_sensors = all_readings_df.filter(col('temperature') > 26) \
    .select('sensor_id', 'temperature') \
    .distinct() \
    .orderBy(col('temperature').desc())

print("=== High Temperature Readings ===")
high_temp_sensors.show()

TypeError: 'int' object is not callable

---
## 4. Data Pipeline Design

### Functional Data Pipeline Principles

1. **Immutability**: Setiap stage menghasilkan data baru
2. **Composability**: Pipeline adalah komposisi fungsi
3. **Declarative**: "What" bukan "how"
4. **Idempotency**: Hasil sama untuk input yang sama

### ETL Pipeline dengan Functional Approach

In [9]:
# Sample raw data
raw_sales_data = [
    {"order_id": "001", "product": "laptop", "amount": "1200.50", "status": "completed"},
    {"order_id": "002", "product": "mouse", "amount": "25.99", "status": "completed"},
    {"order_id": "003", "product": "keyboard", "amount": "75.00", "status": "pending"},
    {"order_id": "004", "product": "laptop", "amount": "invalid", "status": "completed"},
    {"order_id": "005", "product": "monitor", "amount": "350.00", "status": "completed"},
]

# EXTRACT
def extract_data(data):
    """Extract stage - pure function"""
    return spark.createDataFrame(data)

# TRANSFORM - Compose multiple transformations
def clean_amount(df):
    """Remove invalid amounts"""
    from pyspark.sql.functions import col
    return df.filter(col('amount').rlike('^[0-9]+\.?[0-9]*$'))

def parse_amount(df):
    """Convert amount to float"""
    from pyspark.sql.functions import col
    return df.withColumn('amount', col('amount').cast('float'))

def add_category(df):
    """Add product category"""
    from pyspark.sql.functions import when
    return df.withColumn('category',
        when(col('product') == 'laptop', 'Electronics')
        .when(col('product') == 'monitor', 'Electronics')
        .otherwise('Accessories')
    )

def filter_completed(df):
    """Filter only completed orders"""
    return df.filter(col('status') == 'completed')

# Compose transformations functionally
def compose(*functions):
    """Functional composition helper"""
    return reduce(lambda f, g: lambda x: g(f(x)), functions)

# Build pipeline
transform_pipeline = compose(
    clean_amount,
    parse_amount,
    add_category,
    filter_completed
)

# LOAD - could write to database, file, etc.
def load_data(df):
    """Load stage"""
    return df

# Execute ETL Pipeline
print("=== ETL Pipeline Execution ===\n")
raw_df = extract_data(raw_sales_data)

print("Raw data:")
raw_df.show()

transformed_df = transform_pipeline(raw_df)

print("Transformed data:")
transformed_df.show()

# Analytics
print("=== Analytics ===")
transformed_df.groupBy('category').agg(
    spark_sum('amount').alias('total_sales'),
    count('*').alias('order_count')
).show()

=== ETL Pipeline Execution ===

Raw data:


+-------+--------+--------+---------+
| amount|order_id| product|   status|
+-------+--------+--------+---------+
|1200.50|     001|  laptop|completed|
|  25.99|     002|   mouse|completed|
|  75.00|     003|keyboard|  pending|
|invalid|     004|  laptop|completed|
| 350.00|     005| monitor|completed|
+-------+--------+--------+---------+

Transformed data:


+------+--------+-------+---------+-----------+
|amount|order_id|product|   status|   category|
+------+--------+-------+---------+-----------+
|1200.5|     001| laptop|completed|Electronics|
| 25.99|     002|  mouse|completed|Accessories|
| 350.0|     005|monitor|completed|Electronics|
+------+--------+-------+---------+-----------+

=== Analytics ===


TypeError: 'int' object is not callable

### Pipeline Error Handling (Functional Way)

In [10]:
# Functional error handling with Either/Result pattern
from typing import Union, Callable, Any
from dataclasses import dataclass

@dataclass
class Success:
    value: Any
    
@dataclass
class Failure:
    error: str

Result = Union[Success, Failure]

def safe_transform(func: Callable, error_msg: str) -> Callable:
    """Wrapper for safe transformations"""
    def wrapper(df):
        try:
            result = func(df)
            return Success(result)
        except Exception as e:
            return Failure(f"{error_msg}: {str(e)}")
    return wrapper

# Use in pipeline
def run_safe_pipeline(df):
    """Run pipeline with error handling"""
    steps = [
        (clean_amount, "Clean amount failed"),
        (parse_amount, "Parse amount failed"),
        (add_category, "Add category failed"),
    ]
    
    current = Success(df)
    
    for func, error_msg in steps:
        if isinstance(current, Failure):
            return current
        
        safe_func = safe_transform(func, error_msg)
        current = safe_func(current.value)
    
    return current

# Execute
result = run_safe_pipeline(raw_df)

if isinstance(result, Success):
    print("Pipeline succeeded!")
    result.value.show()
else:
    print(f"Pipeline failed: {result.error}")

Pipeline succeeded!
+------+--------+--------+---------+-----------+
|amount|order_id| product|   status|   category|
+------+--------+--------+---------+-----------+
|1200.5|     001|  laptop|completed|Electronics|
| 25.99|     002|   mouse|completed|Accessories|
|  75.0|     003|keyboard|  pending|Accessories|
| 350.0|     005| monitor|completed|Electronics|
+------+--------+--------+---------+-----------+



---
## 5. Real World Case Studies

### Case Study 1: E-Commerce Recommendation System

**Problem**: Recommend products berdasarkan user behavior
**Solution**: Functional pipeline untuk collaborative filtering

In [11]:
# User-Product interaction data
user_interactions = [
    {"user_id": "U1", "product_id": "P1", "rating": 5},
    {"user_id": "U1", "product_id": "P2", "rating": 4},
    {"user_id": "U2", "product_id": "P1", "rating": 5},
    {"user_id": "U2", "product_id": "P3", "rating": 3},
    {"user_id": "U3", "product_id": "P2", "rating": 4},
    {"user_id": "U3", "product_id": "P3", "rating": 5},
    {"user_id": "U4", "product_id": "P1", "rating": 3},
    {"user_id": "U4", "product_id": "P4", "rating": 4},
]

interactions_df = spark.createDataFrame(user_interactions)

# Functional approach to compute product similarity
from pyspark.sql.functions import collect_list, size, array_intersect

# Step 1: Group users by product
product_users = interactions_df.groupBy('product_id').agg(
    collect_list('user_id').alias('users')
)

# Step 2: Self-join to find product pairs
product_pairs = product_users.alias('p1').join(
    product_users.alias('p2'),
    col('p1.product_id') < col('p2.product_id')
)

# Step 3: Calculate similarity (common users)
product_similarity = product_pairs.select(
    col('p1.product_id').alias('product_1'),
    col('p2.product_id').alias('product_2'),
    size(array_intersect(col('p1.users'), col('p2.users'))).alias('common_users')
).filter(col('common_users') > 0)

print("=== Product Similarity Matrix ===")
product_similarity.orderBy(col('common_users').desc()).show()

# Recommendation function (pure function)
def recommend_products(user_id: str, interactions_df, similarity_df, top_n=2):
    """Recommend products for a user"""
    # Products user already rated
    user_products = interactions_df.filter(col('user_id') == user_id) \
        .select('product_id').rdd.flatMap(lambda x: x).collect()
    
    if not user_products:
        return []
    
    # Find similar products
    similar = similarity_df.filter(
        (col('product_1').isin(user_products)) | 
        (col('product_2').isin(user_products))
    )
    
    # Extract recommendations
    recommendations = similar.select(
        expr(f"CASE WHEN product_1 IN {tuple(user_products)} THEN product_2 ELSE product_1 END").alias('recommended'),
        'common_users'
    ).filter(~col('recommended').isin(user_products)) \
     .orderBy(col('common_users').desc()) \
     .limit(top_n)
    
    return recommendations.select('recommended').rdd.flatMap(lambda x: x).collect()

# Test recommendations
for user in ['U1', 'U2', 'U3']:
    recs = recommend_products(user, interactions_df, product_similarity)
    print(f"\nRecommendations for {user}: {recs}")

=== Product Similarity Matrix ===


+---------+---------+------------+
|product_1|product_2|common_users|
+---------+---------+------------+
|       P2|       P3|           1|
|       P1|       P2|           1|
|       P1|       P3|           1|
|       P1|       P4|           1|
+---------+---------+------------+




Recommendations for U1: ['P3', 'P3']



Recommendations for U2: ['P2', 'P2']



Recommendations for U3: ['P1', 'P1']


### Case Study 2: Real-Time Fraud Detection

**Problem**: Detect suspicious transactions in real-time
**Solution**: Stream processing dengan functional rules engine

In [12]:
# Transaction data
transactions = [
    {"trans_id": "T1", "user_id": "U1", "amount": 50.0, "location": "Jakarta"},
    {"trans_id": "T2", "user_id": "U1", "amount": 5000.0, "location": "Jakarta"},
    {"trans_id": "T3", "user_id": "U2", "amount": 100.0, "location": "Bandung"},
    {"trans_id": "T4", "user_id": "U1", "amount": 3000.0, "location": "Singapore"},
    {"trans_id": "T5", "user_id": "U3", "amount": 200.0, "location": "Surabaya"},
    {"trans_id": "T6", "user_id": "U2", "amount": 8000.0, "location": "Bandung"},
]

trans_df = spark.createDataFrame(transactions)

# Functional fraud detection rules
def detect_high_amount(df, threshold=1000.0):
    """Rule: Flag high amount transactions"""
    return df.withColumn('high_amount', col('amount') > threshold)

def detect_location_change(df, user_locations):
    """Rule: Flag location change (simplified)"""
    # In real system, this would check against user's usual locations
    from pyspark.sql.functions import lit
    return df.withColumn('location_change', 
                        col('location').isin(['Singapore', 'Malaysia']))

def calculate_risk_score(df):
    """Rule: Calculate risk score"""
    return df.withColumn('risk_score',
        (col('high_amount').cast('int') * 50 + 
         col('location_change').cast('int') * 30)
    )

def flag_fraud(df, threshold=50):
    """Rule: Flag as fraud if risk score exceeds threshold"""
    return df.withColumn('is_fraud', col('risk_score') >= threshold)

# Compose fraud detection pipeline
fraud_detection_pipeline = compose(
    lambda df: detect_high_amount(df, 1000.0),
    lambda df: detect_location_change(df, {}),
    calculate_risk_score,
    lambda df: flag_fraud(df, 50)
)

# Execute detection
detected = fraud_detection_pipeline(trans_df)

print("=== Fraud Detection Results ===")
detected.select('trans_id', 'user_id', 'amount', 'location', 'risk_score', 'is_fraud').show()

print("\n=== Flagged Transactions ===")
detected.filter(col('is_fraud') == True).show()

=== Fraud Detection Results ===


+--------+-------+------+---------+----------+--------+
|trans_id|user_id|amount| location|risk_score|is_fraud|
+--------+-------+------+---------+----------+--------+
|      T1|     U1|  50.0|  Jakarta|         0|   false|
|      T2|     U1|5000.0|  Jakarta|        50|    true|
|      T3|     U2| 100.0|  Bandung|         0|   false|
|      T4|     U1|3000.0|Singapore|        80|    true|
|      T5|     U3| 200.0| Surabaya|         0|   false|
|      T6|     U2|8000.0|  Bandung|        50|    true|
+--------+-------+------+---------+----------+--------+


=== Flagged Transactions ===
+------+---------+--------+-------+-----------+---------------+----------+--------+
|amount| location|trans_id|user_id|high_amount|location_change|risk_score|is_fraud|
+------+---------+--------+-------+-----------+---------------+----------+--------+
|5000.0|  Jakarta|      T2|     U1|       true|          false|        50|    true|
|3000.0|Singapore|      T4|     U1|       true|           true|        80

### Case Study 3: Log Analytics for Microservices

**Problem**: Analyze distributed logs dari multiple services
**Solution**: MapReduce untuk aggregate metrics

In [13]:
# Distributed microservice logs
microservice_logs = [
    {"timestamp": "2024-01-15T10:00:00", "service": "auth", "endpoint": "/login", "duration_ms": 45, "status": 200},
    {"timestamp": "2024-01-15T10:00:01", "service": "api", "endpoint": "/users", "duration_ms": 120, "status": 200},
    {"timestamp": "2024-01-15T10:00:02", "service": "auth", "endpoint": "/login", "duration_ms": 50, "status": 200},
    {"timestamp": "2024-01-15T10:00:03", "service": "api", "endpoint": "/products", "duration_ms": 200, "status": 500},
    {"timestamp": "2024-01-15T10:00:04", "service": "payment", "endpoint": "/charge", "duration_ms": 300, "status": 200},
    {"timestamp": "2024-01-15T10:00:05", "service": "auth", "endpoint": "/logout", "duration_ms": 30, "status": 200},
    {"timestamp": "2024-01-15T10:00:06", "service": "api", "endpoint": "/products", "duration_ms": 180, "status": 200},
    {"timestamp": "2024-01-15T10:00:07", "service": "payment", "endpoint": "/charge", "duration_ms": 5000, "status": 504},
]

logs_df = spark.createDataFrame(microservice_logs)

# Functional analytics pipeline
print("=== Service Performance Metrics ===")
service_metrics = logs_df.groupBy('service').agg(
    count('*').alias('request_count'),
    spark_round(avg('duration_ms'), 2).alias('avg_duration_ms'),
    spark_sum((col('status') >= 400).cast('int')).alias('error_count')
)
service_metrics.show()

print("=== Endpoint Analysis ===")
endpoint_metrics = logs_df.groupBy('service', 'endpoint').agg(
    count('*').alias('hits'),
    spark_round(avg('duration_ms'), 2).alias('avg_duration')
).orderBy(col('avg_duration').desc())
endpoint_metrics.show()

print("=== SLA Violations (>1000ms) ===")
sla_violations = logs_df.filter(col('duration_ms') > 1000) \
    .select('service', 'endpoint', 'duration_ms', 'status')
sla_violations.show()

# Functional composition: Complex query
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

print("=== Top 2 Slowest Endpoints per Service ===")
window_spec = Window.partitionBy('service').orderBy(col('duration_ms').desc())

slowest_endpoints = logs_df.withColumn('rank', row_number().over(window_spec)) \
    .filter(col('rank') <= 2) \
    .select('service', 'endpoint', 'duration_ms', 'rank')

slowest_endpoints.show()

=== Service Performance Metrics ===


TypeError: 'int' object is not callable

---
## 6. Best Practices dalam Production

### 1. Immutability dan Data Versioning

In [14]:
# Example: Data versioning with immutability
from datetime import datetime

def create_versioned_dataset(df, version):
    """Create immutable versioned dataset"""
    from pyspark.sql.functions import lit
    return df.withColumn('version', lit(version)) \
             .withColumn('created_at', lit(datetime.now().isoformat()))

# Original dataset
original_data = spark.createDataFrame([
    {"id": 1, "name": "Alice", "score": 85},
    {"id": 2, "name": "Bob", "score": 90},
])

# Version 1
v1 = create_versioned_dataset(original_data, "v1")
print("=== Version 1 ===")
v1.show()

# Version 2 with updates (new immutable dataset)
updated_data = spark.createDataFrame([
    {"id": 1, "name": "Alice", "score": 88},  # Updated
    {"id": 2, "name": "Bob", "score": 90},
    {"id": 3, "name": "Charlie", "score": 92},  # New
])

v2 = create_versioned_dataset(updated_data, "v2")
print("=== Version 2 ===")
v2.show()

# Both versions exist independently (immutability)
# Dapat rollback ke v1 kapan saja

=== Version 1 ===


+---+-----+-----+-------+--------------------+
| id| name|score|version|          created_at|
+---+-----+-----+-------+--------------------+
|  1|Alice|   85|     v1|2026-05-18T10:29:...|
|  2|  Bob|   90|     v1|2026-05-18T10:29:...|
+---+-----+-----+-------+--------------------+

=== Version 2 ===
+---+-------+-----+-------+--------------------+
| id|   name|score|version|          created_at|
+---+-------+-----+-------+--------------------+
|  1|  Alice|   88|     v2|2026-05-18T10:29:...|
|  2|    Bob|   90|     v2|2026-05-18T10:29:...|
|  3|Charlie|   92|     v2|2026-05-18T10:29:...|
+---+-------+-----+-------+--------------------+



### 2. Pure Functions dan Testability

In [15]:
# Pure functions are easily testable

def calculate_discount(amount: float, customer_type: str) -> float:
    """Pure function: same input -> same output"""
    discount_rates = {
        'premium': 0.20,
        'regular': 0.10,
        'new': 0.05
    }
    rate = discount_rates.get(customer_type, 0)
    return amount * (1 - rate)

def apply_business_rules(df):
    """Pure transformation"""
    from pyspark.sql.functions import udf
    from pyspark.sql.types import FloatType
    
    discount_udf = udf(calculate_discount, FloatType())
    
    return df.withColumn(
        'final_amount',
        discount_udf(col('amount'), col('customer_type'))
    )

# Testing is straightforward
orders = spark.createDataFrame([
    {"order_id": "O1", "amount": 100.0, "customer_type": "premium"},
    {"order_id": "O2", "amount": 100.0, "customer_type": "regular"},
    {"order_id": "O3", "amount": 100.0, "customer_type": "new"},
])

result = apply_business_rules(orders)
print("=== Business Rules Applied ===")
result.show()

# Unit test example (can run separately)
assert calculate_discount(100, 'premium') == 80.0
assert calculate_discount(100, 'regular') == 90.0
assert calculate_discount(100, 'new') == 95.0
print("✓ All tests passed!")

/Users/mac/achluky.github.io/pbf/week11/.venv/lib/python3.11/site-packages/pyspark/sql/udf.py:134: UserWarning: Cannot infer the eval type from type hints. 
  warnings.warn("Cannot infer the eval type from type hints. ", UserWarning)


=== Business Rules Applied ===


+------+-------------+--------+------------+
|amount|customer_type|order_id|final_amount|
+------+-------------+--------+------------+
| 100.0|      premium|      O1|        80.0|
| 100.0|      regular|      O2|        90.0|
| 100.0|          new|      O3|        95.0|
+------+-------------+--------+------------+

✓ All tests passed!


### 3. Composability dan Reusability

In [16]:
# Build library of reusable transformations

# Generic transformations
def add_timestamp(df):
    """Add processing timestamp"""
    from pyspark.sql.functions import current_timestamp
    return df.withColumn('processed_at', current_timestamp())

def add_hash_id(df, columns):
    """Add hash ID from specified columns"""
    from pyspark.sql.functions import md5, concat_ws
    return df.withColumn('hash_id', md5(concat_ws('|', *columns)))

def filter_by_date(df, date_column, start_date):
    """Filter by date range"""
    return df.filter(col(date_column) >= start_date)

# Compose into custom pipeline
def create_audit_pipeline(*transformations):
    """Create auditable data pipeline"""
    def pipeline(df):
        # Add audit fields at start
        df = add_timestamp(df)
        
        # Apply transformations
        for transform in transformations:
            df = transform(df)
        
        return df
    return pipeline

# Example usage
sample_data = spark.createDataFrame([
    {"user_id": "U1", "action": "login", "date": "2024-01-15"},
    {"user_id": "U2", "action": "purchase", "date": "2024-01-16"},
])

# Compose pipeline
my_pipeline = create_audit_pipeline(
    lambda df: add_hash_id(df, ['user_id', 'action']),
    lambda df: df.withColumn('action_upper', expr('upper(action)'))
)

result = my_pipeline(sample_data)
print("=== Composed Pipeline Result ===")
result.show(truncate=False)

=== Composed Pipeline Result ===


+--------+----------+-------+--------------------------+--------------------------------+------------+
|action  |date      |user_id|processed_at              |hash_id                         |action_upper|
+--------+----------+-------+--------------------------+--------------------------------+------------+
|login   |2024-01-15|U1     |2026-05-18 10:29:45.434494|98eff0f07c4b19f8e5a8b9448c7bb50b|LOGIN       |
|purchase|2024-01-16|U2     |2026-05-18 10:29:45.434494|5d35fc9dc5e9ec2df2f0ed29ddf7b46f|PURCHASE    |
+--------+----------+-------+--------------------------+--------------------------------+------------+



### 4. Error Handling dan Logging

In [ ]:
# Functional error handling in production

from typing import Tuple
import logging

def validate_and_transform(df, validation_rules):
    """Validate data and separate valid/invalid records"""
    
    # Start with all records as valid
    valid_df = df
    invalid_records = []
    
    for rule_name, rule_func in validation_rules.items():
        # Apply validation
        valid_df = valid_df.filter(rule_func())
        
        # Collect invalid (for logging/debugging)
        invalid_count = df.count() - valid_df.count()
        if invalid_count > 0:
            invalid_records.append({
                'rule': rule_name,
                'failed_count': invalid_count
            })
    
    return valid_df, invalid_records

# Example validation rules
validation_rules = {
    'amount_positive': lambda: col('amount') > 0,
    'amount_reasonable': lambda: col('amount') < 10000,
}

# Test data with some invalid records
test_data = spark.createDataFrame([
    {"id": 1, "amount": 100.0},   # valid
    {"id": 2, "amount": -50.0},   # invalid: negative
    {"id": 3, "amount": 15000.0}, # invalid: too high
    {"id": 4, "amount": 200.0},   # valid
])

valid, invalid = validate_and_transform(test_data, validation_rules)

print("=== Valid Records ===")
valid.show()

print("\n=== Validation Report ===")
for record in invalid:
    print(f"Rule '{record['rule']}': {record['failed_count']} records failed")

### 5. Performance Optimization

In [ ]:
# Functional programming best practices for performance

# 1. Avoid wide transformations when possible
# 2. Use narrow transformations (map, filter) 
# 3. Minimize shuffles
# 4. Cache intermediate results when reused

# Example: Optimize with caching
large_dataset = spark.range(1000000).toDF("id") \
    .withColumn("value", expr("id * 2"))

# Bad: Multiple actions without caching
print("=== Without Caching ===")
# Each of these triggers full computation
# count1 = large_dataset.filter(col("value") > 1000).count()
# max_val = large_dataset.agg({"value": "max"}).collect()

# Good: Cache when multiple actions needed
print("=== With Caching ===")
filtered = large_dataset.filter(col("value") > 1000).cache()
count1 = filtered.count()
max_val = filtered.agg({"value": "max"}).collect()[0][0]

print(f"Count: {count1}")
print(f"Max: {max_val}")

# Clean up
filtered.unpersist()

# 2. Broadcast small lookup tables
small_lookup = spark.createDataFrame([
    {"code": "A", "description": "Type A"},
    {"code": "B", "description": "Type B"},
])

large_facts = spark.createDataFrame([
    {"id": 1, "code": "A", "value": 100},
    {"id": 2, "code": "B", "value": 200},
    {"id": 3, "code": "A", "value": 150},
])

# Broadcast join (efficient)
from pyspark.sql.functions import broadcast

enriched = large_facts.join(
    broadcast(small_lookup),  # Broadcast small table
    "code"
)

print("\n=== Broadcast Join Result ===")
enriched.show()

### 6. Monitoring dan Observability

In [ ]:
# Add monitoring to functional pipelines

from time import time
from typing import Callable

def monitored_transformation(name: str, transform_func: Callable):
    """Wrapper to monitor transformation performance"""
    def wrapper(df):
        start_time = time()
        input_count = df.count()
        
        # Apply transformation
        result = transform_func(df)
        
        output_count = result.count()
        duration = time() - start_time
        
        # Log metrics
        metrics = {
            'transformation': name,
            'input_records': input_count,
            'output_records': output_count,
            'duration_seconds': round(duration, 2),
            'records_per_second': round(output_count / duration if duration > 0 else 0, 2)
        }
        
        print(f"\n=== Transformation Metrics: {name} ===")
        for key, value in metrics.items():
            print(f"{key}: {value}")
        
        return result
    
    return wrapper

# Example: Monitored pipeline
sample = spark.range(10000).toDF("id")

# Wrap transformations with monitoring
transform1 = monitored_transformation(
    "double_values",
    lambda df: df.withColumn("doubled", col("id") * 2)
)

transform2 = monitored_transformation(
    "filter_large",
    lambda df: df.filter(col("doubled") > 1000)
)

# Execute monitored pipeline
result = transform1(sample)
result = transform2(result)

---
## Summary: Key Takeaways

### Keuntungan FP untuk Big Data Production:

1. **Scalability**: Pure functions mudah di-parallelize
2. **Maintainability**: Code lebih modular dan composable
3. **Testability**: Pure functions mudah di-test
4. **Debugging**: Immutability membuat debugging lebih mudah
5. **Reliability**: Stateless operations lebih reliable di distributed systems

### Best Practices Checklist:

✅ Gunakan immutable data structures  
✅ Prefer pure functions  
✅ Compose small, focused transformations  
✅ Implement proper error handling  
✅ Add monitoring dan logging  
✅ Cache strategic intermediate results  
✅ Write unit tests for transformations  
✅ Document data lineage  
✅ Version your datasets  
✅ Optimize for lazy evaluation  

### Tools dan Frameworks:

- **Apache Spark**: Distributed computing dengan FP
- **Kafka Streams**: Stream processing dengan FP
- **Flink**: Stream & batch processing
- **Beam**: Unified programming model

---
## Latihan

### Exercise 1: Build ETL Pipeline
Buat functional ETL pipeline untuk memproses customer data:
- Extract dari source
- Clean invalid records
- Transform dengan business rules
- Aggregate metrics
- Handle errors gracefully

### Exercise 2: Real-Time Analytics
Implementasikan real-time analytics dashboard untuk monitoring aplikasi:
- Stream processing untuk metrics
- Windowed aggregations
- Alert detection
- Functional composition

### Exercise 3: Data Quality Framework
Buat reusable data quality framework:
- Define validation rules sebagai pure functions
- Compose validation pipeline
- Generate quality reports
- Track data lineage

### Exercise 4: Performance Optimization
Optimize existing pipeline:
- Identify bottlenecks
- Apply caching strategies
- Optimize joins
- Measure improvements

---
## Resources

### Documentation:
- [Apache Spark Programming Guide](https://spark.apache.org/docs/latest/programming-guide.html)
- [Structured Streaming Guide](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html)
- [PySpark API Reference](https://spark.apache.org/docs/latest/api/python/)

### Books:
- "Functional Programming in Scala" - Paul Chiusano & Rúnar Bjarnason
- "Learning Spark" - Jules S. Damji et al.
- "Streaming Systems" - Tyler Akidau et al.

### Courses:
- Big Data Analysis with Scala and Spark (Coursera)
- Apache Spark Essential Training (LinkedIn Learning)

---

**End of Notebook**

*Created for: Functional Programming in Big Data Course*  
*Topic: Real-World Applications and Best Practices*

In [ ]:
# Cleanup
spark.stop()
print("Spark session stopped.")